# Google Colab Runner: Vietnamese Medical NER & Candidate Linking Pipeline
This notebook mounts Google Drive, clones the repository, installs the dependencies with CUDA GPU support, links the GGUF model from Google Drive, and executes the pipeline on the T4 GPU.

In [ ]:
# 1. Mount Google Drive to load/save the GGUF model
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup the Workspace
Clone the repository and install requirements.

In [ ]:
import os

# Clone the repo
!git clone https://github.com/tmanhococ/VT_MedicalRetrieval.git
%cd VT_MedicalRetrieval

# Install requirements
!pip install -r requirements.txt

# Force-reinstall llama-cpp-python with CUDA support (takes ~1-2 mins to compile)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir

## 3. Configure Model Path from Google Drive
This checks if the model exists in your Google Drive (in `MedicalNER/models/`). If not, it downloads it once and saves it there so you do not have to download it again in future sessions.

In [ ]:
import os
import urllib.request

drive_model_dir = "/content/drive/MyDrive/MedicalNER/models"
os.makedirs(drive_model_dir, exist_ok=True)
drive_model_path = os.path.join(drive_model_dir, "Qwen2.5-7B-Instruct-Q4_K_M.gguf")

# Download the model to Drive if not present
if not os.path.exists(drive_model_path):
    print("Model not found in Google Drive. Downloading from HuggingFace (4.7 GB)...")
    url = "https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    # Helper to download with progress indicator
    def progress(block_num, block_size, total_size):
        percent = (block_num * block_size / total_size) * 100
        print(f"\rDownloading: {percent:.2f}%", end="")
    urllib.request.urlretrieve(url, drive_model_path, progress)
    print("\nDownload finished!")
else:
    print("Model found in Google Drive. Reusing it.")

# Link the GDrive model path to local data/models directory
os.makedirs("data/models", exist_ok=True)
local_link_path = "data/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
if not os.path.exists(local_link_path):
    os.symlink(drive_model_path, local_link_path)
    print("Symlink created to Google Drive model.")
else:
    print("Model link already exists.")

## 4. Run the Pipeline on GPU
We run the pipeline. By default, since `COLAB_GPU` is active, config.py will set `GPU_LAYERS_OFFLOAD = -1` (offload all layers to T4 GPU).

In [ ]:
# Run pipeline
os.environ["COLAB_GPU"] = "1"
os.environ["GPU_LAYERS_OFFLOAD"] = "-1"

!python -u main.py

## 5. Download the Submission ZIP

In [ ]:
from google.colab import files
files.download('output.zip')